# NewsRetrieval — Adquisición, normalización y representaciones vectoriales

Este notebook recorre el procesamiento de noticias desde las fuentes y el corpus original hasta las matrices VSM. Se reutilizan los archivos y funciones existentes del proyecto.

**Flujo:** Fuentes de noticias → Corpus original → Normalización → Corpus normalizado → N-gramas → Representaciones VSM.

## 2. Adquisición y corpus original

La unidad documental es una noticia. Las tablas siguientes resumen las fuentes configuradas, el corpus local y la distribución de noticias; no realizan nuevas descargas.

In [1]:
from collections import Counter
from html import escape
from pathlib import Path
from urllib.parse import urlsplit
import json
import joblib

from collector import CORPUS, CATEGORIES, FEEDS, WIKINEWS, read_corpus
from retrieval import NORMALIZED, VECTORS, STOP_POS, clean_text, load_nlp, normalize_text


class NotebookHTML:
    """Objeto sencillo que Jupyter presenta como una tabla HTML."""
    def __init__(self, html):
        self.html = html

    def _repr_html_(self):
        return self.html


def html_table(headers, rows, caption=None):
    style = ('border-collapse:collapse; width:max-content; min-width:100%; '
             'font-size:13px; font-family:Arial,sans-serif;')
    th = ('position:sticky; top:0; background:#eaf0f6; color:#172b4d; '
          'font-weight:600; text-align:left;')
    td = 'vertical-align:top; text-align:left;'
    pieces = ['<div style="max-width:100%; overflow-x:auto; margin:8px 0 18px 0;">']
    if caption:
        pieces.append(f'<div style="font-weight:600; margin:4px 0 6px 0;">{escape(str(caption))}</div>')
    pieces.append(f'<table style="{style}"><thead><tr>')
    pieces.extend(f'<th style="{th} padding:7px 9px; border:1px solid #c7d1dc;">{escape(str(h))}</th>' for h in headers)
    pieces.append('</tr></thead><tbody>')
    for row in rows:
        pieces.append('<tr>')
        for value in row:
            content = escape(str(value))
            pieces.append(f'<td style="{td} padding:6px 9px; border:1px solid #d8dee6;">'
                          f'<div style="max-width:290px; max-height:4.5em; overflow:hidden; '
                          f'overflow-wrap:anywhere; line-height:1.45;">{content}</div></td>')
        pieces.append('</tr>')
    pieces.append('</tbody></table></div>')
    return ''.join(pieces)


def html_block(title, body):
    return (f'<section style="margin:10px 0 18px 0;"><h4 style="margin:8px 0;">{escape(str(title))}</h4>'
            f'{body}</section>')


def html_text(title, text):
    return (f'<section style="margin:10px 0 16px 0;"><h4 style="margin:6px 0;">{escape(str(title))}</h4>'
            f'<div style="max-height:130px; overflow:auto; white-space:pre-wrap; overflow-wrap:anywhere; '
            f'border:1px solid #d8dee6; background:#fafbfc; padding:10px;">{escape(str(text))}</div></section>')


def output(*parts):
    return NotebookHTML(''.join(part._repr_html_() if isinstance(part, NotebookHTML) else str(part) for part in parts))


def clipped(value, limit=170):
    value = str(value)
    return value if len(value) <= limit else value[:limit].rstrip() + '…'

raw_rows = read_corpus(CORPUS)
normalized_rows = read_corpus(NORMALIZED)
raw_by_id = {row['id']: row for row in raw_rows}
normalized_by_id = {row['id']: row for row in normalized_rows}


In [2]:
source_rows = []
gv_categories = [category for category, urls in FEEDS.items()
                 if any(urlsplit(url).netloc.endswith('globalvoices.org') for url in urls)]
eff_categories = [category for category, urls in FEEDS.items()
                  if any(urlsplit(url).netloc.endswith('eff.org') for url in urls)]
source_rows = [
    ['Wikinews', ', '.join(category for category in CATEGORIES if category in WIKINEWS), 'API MediaWiki'],
    ['Global Voices', ', '.join(category for category in CATEGORIES if category in gv_categories), 'RSS/Atom'],
    ['EFF', ', '.join(category for category in CATEGORIES if category in eff_categories), 'RSS/Atom'],
]
source_table = html_table(['Fuente', 'Categorías', 'Mecanismo de adquisición'], source_rows)

required_columns = ['id', 'title', 'description/body', 'source', 'category', 'date', 'url', 'retrieved_at']
original_sample = [[row[key] if key not in {'title', 'description/body', 'url'} else clipped(row[key], 160)
                    for key in required_columns] for row in raw_rows[:8]]
original_table = html_table(required_columns, original_sample, 'Noticias originales (8 filas)')
source_counts = html_table(['Fuente', 'Noticias'], sorted(Counter(r['source'] for r in raw_rows).items()))
category_counts = html_table(['Categoría', 'Noticias'], sorted(Counter(r['category'] for r in raw_rows).items()))
year_counts = html_table(['Año de publicación', 'Noticias'],
                          sorted(Counter(r['date'][:4] for r in raw_rows).items()))
output(html_block('Fuentes configuradas', source_table),
       html_block('Corpus original', original_table),
       html_block('Distribución por fuente', source_counts),
       html_block('Distribución por categoría', category_counts),
       html_block('Distribución por año', year_counts))

<tablas HTML de Jupyter>

## 3. Tokenización y análisis gramatical

Se observa un registro real. spaCy aporta los tokens, las etiquetas POS y los lemas; la marca de conservación se obtiene comparando con la salida de `normalize_text`, sin repetir sus reglas. El texto original y su normalización se muestran debajo de la tabla.

In [3]:
sample = next(row for row in raw_rows if len(row['description/body'].split()) >= 100)
analysis_text = sample['title'] + '. ' + sample['description/body']
nlp = load_nlp()
doc = nlp(clean_text(analysis_text))
normalized_tokens = normalize_text(analysis_text).split()
normalized_position = 0
token_rows = []
for token in doc:
    if token.is_space:
        continue
    lemma = token.lemma_.lower().strip()
    kept = normalized_position < len(normalized_tokens) and lemma == normalized_tokens[normalized_position]
    token_rows.append([token.text, token.pos_, lemma, 'Sí' if kept else 'No'])
    if kept:
        normalized_position += 1

original_text_html = html_text('Texto original', analysis_text)
token_table = html_table(['Token original', 'POS', 'Lema', '¿Se conserva?'], token_rows[:20],
                         'Primeros 20 tokens')
normalized_text_html = html_text('Texto normalizado', normalize_text(analysis_text))
output(original_text_html, token_table, normalized_text_html)

Token original,POS,Lema,¿Se conserva?
In,ADP,in,No
Memoriam,PROPN,memoriam,Sí
:,PUNCT,:,No
Barry,PROPN,barry,Sí
Steinhardt,PROPN,steinhardt,Sí
.,PUNCT,.,No
Barry,PROPN,barry,Sí
Steinhardt,PROPN,steinhardt,Sí
",",PUNCT,",",No
a,DET,a,No


## 4. Corpus normalizado

Las columnas conservan el orden y los metadatos del corpus original. Para esta presentación, `title` y `description/body` se mapean a `normalized_title` y `normalized_body`; el TSV almacenado no se modifica.

In [4]:
normalized_sample = [[row['id'], clipped(row['normalized_title'], 140),
                      clipped(row['normalized_body'], 180), row['source'], row['category'],
                      row['date'], clipped(row['url'], 110), row['retrieved_at']]
                     for row in normalized_rows[:8]]
normalized_table = html_table(required_columns, normalized_sample, 'Corpus normalizado (8 filas)')
comparison_rows = [[row['id'], clipped(raw_by_id[row['id']]['title'], 150),
                    clipped(row['normalized_title'], 150)] for row in normalized_rows[:5]]
comparison_table = html_table(['ID', 'Título original', 'Título normalizado'], comparison_rows)
output(html_block('Noticias normalizadas', normalized_table),
       html_block('Comparación de títulos (5 artículos)', comparison_table))

<tablas HTML de Jupyter>

## 5. N-gramas y probabilidades empíricas

Los conteos se calculan sobre títulos y descripciones normalizados, pero cada campo de cada noticia se procesa por separado. Así ningún n-grama cruza límites entre noticias o campos. Estas probabilidades son una demostración complementaria; no forman parte de las matrices VSM exigidas.

In [5]:
unigram_counts = Counter()
bigram_counts = Counter()
bigram_context_counts = Counter()
trigram_counts = Counter()
trigram_context_counts = Counter()
total_tokens = 0

for row in normalized_rows:
    for field in ('normalized_title', 'normalized_body'):
        tokens = row[field].split()
        total_tokens += len(tokens)
        unigram_counts.update(tokens)
        bigram_counts.update(zip(tokens, tokens[1:]))
        bigram_context_counts.update(tokens[:-1])
        trigrams = list(zip(tokens, tokens[1:], tokens[2:]))
        trigram_counts.update(trigrams)
        trigram_context_counts.update(zip(tokens, tokens[1:]))

unigram_rows = [[word, count, f'{count / total_tokens:.6f}']
                for word, count in unigram_counts.most_common(20)]
bigram_rows = [[w1, w2, count, bigram_context_counts[w1],
                f'{count / bigram_context_counts[w1]:.6f}']
               for (w1, w2), count in bigram_counts.most_common(20)]
trigram_rows = [[w1, w2, w3, count, trigram_context_counts[(w1, w2)],
                 f'{count / trigram_context_counts[(w1, w2)]:.6f}']
                for (w1, w2, w3), count in trigram_counts.most_common(20)]

unigram_table = html_table(['Palabra', 'Frecuencia', 'Probabilidad P(w)'], unigram_rows)
bigram_table = html_table(['Palabra 1', 'Palabra 2', 'Frecuencia del bigrama',
                           'Frecuencia del contexto', 'P(w2 | w1)'], bigram_rows)
trigram_table = html_table(['Palabra 1', 'Palabra 2', 'Palabra 3', 'Frecuencia del trigrama',
                            'Frecuencia del contexto', 'P(w3 | w1,w2)'], trigram_rows)
output(html_block('A. Unigramas — P(w) = frecuencia(w) / total de tokens', unigram_table),
       html_block('B. Bigramas — P(w2 | w1)', bigram_table),
       html_block('C. Trigramas — P(w3 | w1,w2)', trigram_table))

Palabra,Frecuencia,Probabilidad P(w)
be,20757,0.045012
to,10667,0.023132
have,7528,0.016325
not,3237,0.007020
say,3105,0.006733
this,2884,0.006254
will,2779,0.006026
do,1858,0.004029
which,1834,0.003977
new,1752,0.003799


## 6. Representaciones VSM

Se cargan los vectorizadores y matrices dispersas guardados en `results/vectors.joblib`. Cada tabla muestra un recorte real: hasta 20 términos por fila y cinco noticias por columna. Las mismas noticias y características se usan para comparar frecuencia, binaria y TF-IDF.

- **Frecuencia:** cantidad de apariciones del término.
- **Binaria:** 1 si el término aparece en el documento y 0 si no aparece.
- **TF-IDF:** peso relativo del término en el documento y en el corpus; es un peso, no una probabilidad.

TF-IDF es la representación principal para las consultas posteriores.

In [6]:
bundle = joblib.load(VECTORS)

summary_rows = []
for field in ('title', 'body'):
    for n in (1, 2):
        for representation in ('frequency', 'binary', 'tfidf'):
            matrix = bundle['spaces'][f'{field}_{n}_{representation}']['matrix']
            summary_rows.append([field, n, representation, matrix.shape[0], matrix.shape[1]])
summary_table = html_table(['Campo', 'N-grama (n)', 'Representación', 'Documentos', 'Características'],
                           summary_rows)


def choose_matrix_slice(field, n, limit_docs=5, limit_terms=20):
    frequency = bundle['spaces'][f'{field}_{n}_frequency']['matrix']
    vectorizer = bundle['spaces'][f'{field}_{n}_frequency']['vectorizer']
    names = vectorizer.get_feature_names_out()
    document_frequency = frequency.getnnz(axis=0).ravel()
    candidate_features = sorted(range(len(names)),
                                key=lambda j: (-document_frequency[j], names[j]))[:100]
    candidate_rows = frequency[:, candidate_features].getnnz(axis=1).ravel()
    document_indexes = sorted(range(frequency.shape[0]),
                              key=lambda i: (-candidate_rows[i], i))[:limit_docs]
    feature_union = set()
    for i in document_indexes:
        feature_union.update(frequency.getrow(i).indices)
    feature_indexes = sorted(
        feature_union,
        key=lambda j: (
            -sum(frequency[i, j] != 0 for i in document_indexes),
            -sum(frequency[i, j] for i in document_indexes),
            names[j]))[:limit_terms]
    return document_indexes, feature_indexes, names


matrix_sections = []
for field in ('title', 'body'):
    for n in (1, 2):
        document_indexes, feature_indexes, names = choose_matrix_slice(field, n)
        full_shape = bundle['spaces'][f'{field}_{n}_frequency']['matrix'].shape
        document_headers = [f"{bundle['ids'][i]} · {clipped(bundle['documents'][i]['title'], 46)}"
                            for i in document_indexes]
        representation_tables = []
        for representation, label in (
                ('frequency', 'Frecuencia'),
                ('binary', 'Binaria'),
                ('tfidf', 'TF-IDF')):
            matrix = bundle['spaces'][f'{field}_{n}_{representation}']['matrix']
            rows = []
            for j in feature_indexes:
                values = []
                for i in document_indexes:
                    value = matrix[i, j]
                    values.append(f'{value:.4f}' if representation == 'tfidf' else int(value))
                rows.append([names[j]] + values)
            table = html_table(['Término / n-grama'] + document_headers, rows,
                               f'{label} · valores de la matriz')
            representation_tables.append(html_block(label, table))
        matrix_sections.append(html_block(
            f'{"Título" if field == "title" else "Descripción/cuerpo"} · n={n} · recorte {len(feature_indexes)} × {len(document_indexes)} de {full_shape[0]} × {full_shape[1]}',
            ''.join(representation_tables)))

output(html_block('Dimensiones de las 12 matrices VSM', summary_table), *matrix_sections)

Campo,N-grama (n),Representación,Documentos,Características
title,1,frequency,2000,4763
title,1,binary,2000,4763
title,1,tfidf,2000,4763
title,2,frequency,2000,11371
title,2,binary,2000,11371
title,2,tfidf,2000,11371
body,1,frequency,2000,28213
body,1,binary,2000,28213
body,1,tfidf,2000,28213
body,2,frequency,2000,281416
